# 020. LSTM/GRU input/output shape

- return_sequences = False, True 일 때의 output 비교

- return_state = False, True 일 때의 internal state output 비교

- Bidirectional LSTM/GRU 의 output 비교

- LSTM 이 GRU 보다 계산량 많음

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Bidirectional, GRU
import numpy as np
import warnings
warnings.filterwarnings('ignore')

B = 2   # batch size
T = 5   #Time Steps
D = 1   #features
U = 3   #LSTM units -> output

X = np.random.randn(B, T, D)
print(X.shape)

(2, 5, 1)


# LSTM

## return_sequences

- False (default) - last time step 의 output 만 반환
- True - 모든 timestep 의 output 을 모두 반환

In [ ]:
def lstm(return_sequences=False):
    inp = Input(shape=(T, D)) # Timestep Dimesion
    out = LSTM(U, return_sequences=return_sequences)(inp)

    model = Model(inputs=inp, outputs=out)
    return model.predict(X)

print("---- return_sequences=False ----> last timestep 의 output 만 반환")
lstm_out = lstm(return_sequences=False)
print(lstm_out.shape)
print(lstm_out)

print("\n---- return_sequences=True ----> 모든 timestep 별 output 출력")
lstm_out = lstm(return_sequences=True)
print(lstm_out.shape)
print(lstm_out)

---- return_sequences=False ----> last timestep 의 output 만 반환
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 301ms/step
(2, 3)
[[-0.02156628  0.03416369  0.12547764]
 [ 0.02179278  0.00152821  0.043912  ]]

---- return_sequences=True ----> 모든 timestep 별 output 출력
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 177ms/step
(2, 5, 3)
[[[-0.0554461  -0.07277849  0.06394804]
  [-0.14534926 -0.16783756  0.1747409 ]
  [-0.06817476 -0.07880929  0.15295343]
  [ 0.03896358  0.05819116  0.01689683]
  [ 0.07852974  0.1089365  -0.05302533]]

 [[ 0.07713248  0.10629237 -0.10838599]
  [-0.04500329 -0.11654437  0.04823753]
  [ 0.04200197  0.01105481 -0.03633726]
  [ 0.01340881 -0.02769617 -0.00073636]
  [ 0.00598204 -0.02041704  0.00603538]]]


## return_state

- False (default) - output 만 반환

- True - output, last step 의 hidden state, cell state (LSTM 의 경우) 반환

In [ ]:
def lstm(return_state=False):
    inp = Input(shape=(T, D))
    out = LSTM(U, return_state=return_state)(inp)

    model = Model(inputs=inp, outputs=out)

    '''
    h,c는 인코더 디코더 모델이 있을 때, 초기에 h,c을 가져다 쓰는 데 사용
    '''

    if return_state:
        o, h, c = model.predict(X)
        print("o :", o.shape)
        print(o)
        print("h :", h.shape)
        print(h)
        print("c :", c.shape)
        print(c)
    else:
        o = model.predict(X)
        print("o :", o.shape)
        print(o)

print("---- return_state=False ----> outout only")
lstm(return_state=False)
print("\n---- return_state=True ----> outout, hidden state, cell state all")
lstm(return_state=True)

---- return_state=False ----> outout only
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step
o : (2, 3)
[[-1.0692911e-01 -1.2804714e-01 -8.0025522e-03]
 [-2.0610455e-02  5.2714455e-05  1.6767871e-02]]

---- return_state=True ----> outout, hidden state, cell state all
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 174ms/step
o : (2, 3)
[[-0.09475733 -0.14419498 -0.02734525]
 [-0.04734539 -0.02713876  0.01865435]]
h : (2, 3)
[[-0.09475733 -0.14419498 -0.02734525]
 [-0.04734539 -0.02713876  0.01865435]]
c : (2, 3)
[[-0.16373536 -0.26445764 -0.06392401]
 [-0.09531543 -0.0546352   0.03684666]]


# Bidirectional LSTM

- 순방향, 역방향이 concatenate 된 output 출력  

- hidden state, cell state 는 순방향, 역방향 별도 출력

In [ ]:
T, D, U #Time Steps / features /LSTM units -> output

(5, 1, 3)

In [ ]:
def bi_lstm(return_sequences=False, return_state=False):
    inp = Input(shape=(T, D))

    # bidirectional 함수만 추가
    out = Bidirectional(
            LSTM(U, return_state=return_state, return_sequences=return_sequences))(inp)

    model = Model(inputs=inp, outputs=out)

    if return_state:
        o, h1, c1, h2, c2 = model.predict(X)
        print("o :",o.shape)
        print("h1 :", h1.shape)
        print("c1 :", c1.shape)
        print("h2 :", h2.shape)
        print("c2 :", c2.shape)
    else:
        o = model.predict(X)
        print("o :", o.shape)

print("*** 순방향, 역방향이 concatenate ***")
print("---- return_sequences=False ----> last timestep 의 output 만 반환")

# output 이 두배로 나옴 (순방향 + 역방향)
bi_lstm(return_sequences=False, return_state=False)
print()
print("---- return_sequences=True ----> 모든 timestep 별 output 출력")
bi_lstm(return_sequences=True)
print()
print("---- return_sequences=True, return_state=True")
bi_lstm(return_state=True)

*** 순방향, 역방향이 concatenate ***
---- return_sequences=False ----> last timestep 의 output 만 반환
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 339ms/step
o : (2, 6)

---- return_sequences=True ----> 모든 timestep 별 output 출력
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 309ms/step
o : (2, 5, 6)

---- return_sequences=True, return_state=True
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 344ms/step
o : (2, 6)
h1 : (2, 3)
c1 : (2, 3)
h2 : (2, 3)
c2 : (2, 3)


# GRU

- cell state 가 없는 것만 LSTM 과 차이

In [ ]:
def gru(return_sequences=False, return_state=False):
    inp = Input(shape=(T, D))
    out = GRU(U, return_state=return_state, return_sequences=return_sequences)(inp)

    model = Model(inputs=inp, outputs=out)

    if return_state:
        o, h = model.predict(X)
        print("o :", o.shape)
        print("h :", h.shape)
    else:
        o = model.predict(X)
        print("o :", o.shape)

print("---- Many-to-One output ----")
gru(return_sequences=False, return_state=False)
print()
print("---- Many-to-Many output ----")
gru(return_sequences=True)
print()
print("---- Sequence-to-Vector output ----")
gru(return_state=True)

---- Many-to-One output ----
o : (2, 3)

---- Many-to-Many output ----
o : (2, 5, 3)

---- Sequence-to-Vector output ----
o : (2, 3)
h : (2, 3)


# Bidirectional GRU

- cell state 가 없는 것 외에 LSTM 과 동일

In [ ]:
def bi_gru(return_sequences=False, return_state=False):
    inp = Input(shape=(T, D))
    out = Bidirectional(
            GRU(U, return_state=return_state, return_sequences=return_sequences))(inp)

    model = Model(inputs=inp, outputs=out)
    if return_state:
        o, h1, h2 = model.predict(X)
        print("o :", o.shape)
        print("h1 :", h1.shape)
        print("h2 :", h2.shape)
    else:
        o = model.predict(X)
        print("o :", o.shape)

print("---- 순방향, 역방향이 concatenate 된 many-to-one output")
bi_gru(return_sequences=False, return_state=False)
print()
print("---- 순방향, 역방향이 concatenate 된 many-to-many output")
bi_gru(return_sequences=True)
print()
print("---- 순방향, 역방향이 concatenate 된 sequence-to-vector output")
bi_gru(return_state=True)

---- 순방향, 역방향이 concatenate 된 many-to-one output
o : (2, 6)

---- 순방향, 역방향이 concatenate 된 many-to-many output
o : (2, 5, 6)

---- 순방향, 역방향이 concatenate 된 sequence-to-vector output
o : (2, 6)
h1 : (2, 3)
h2 : (2, 3)
